In [ ]:
# Install dependencies into the active kernel
%pip install flatten_json pandas matplotlib

In [ ]:
import pandas as pd
import json
from collections import defaultdict
from flatten_json import flatten
import matplotlib.pyplot as plt
import pathlib

In [ ]:
files = [
    'xclipse_950_2026_07_10.json',
    "adreno_840_2026_07_10.json",
]

In [ ]:
def aggregate_loops_passes(loops):
    results_per_frame = []
    num_loops = len(loops)
    for loop_results in loops:
        for frame_index, frame_results in enumerate(loop_results["per_frame_results"]):
            if frame_index >= len(results_per_frame):
                results_per_frame.append(defaultdict(int))
            results_per_frame[frame_index]['sequence_time_ns'] = frame_results['sequence_time_ns']
            for command_buffer_timings in frame_results["command_buffer_timings"].values():
                for scope_name, scope_timings in command_buffer_timings["scope_timings"].items():
                    # A pass can run multiple times per frame; sum its durations, average over loops.
                    for scope_timing in scope_timings:
                        results_per_frame[frame_index][scope_name] += (
                            scope_timing["end"] - scope_timing["start"]
                        ) / num_loops / 1_000_000  # in ms
            if frame_results["metrics"] is not None:
                for metric_name, metric in frame_results["metrics"].items():
                    # TODO: Flatten this in rust to fan_speed_rpm
                    if metric is not None and metric_name != "timestamp":
                        results_per_frame[frame_index][metric_name] += metric / num_loops
    # TODO: Aggregate CPU timings
    return pd.DataFrame([flatten(x) for x in results_per_frame])


# Load every input file, aggregating its loops/passes into one number per pass per frame.
results = {}
for path in files:
    with open(path, "r") as json_file:
        results[path] = aggregate_loops_passes(json.load(json_file))

# Concat into one frame: (input file, frame) per row, metric per column
full_dataset = pd.concat(results)
full_dataset

In [ ]:
# Print all possible metrics
full_dataset.columns.tolist()

In [ ]:
metrics = full_dataset

# Reshape into sequence time + metric type per row, input file per column
metrics = metrics.reset_index().set_index(['sequence_time_ns', 'level_0']).drop('level_1', axis=1)
metrics = metrics.stack().unstack(1).reset_index()

# From ns to s
metrics['sequence_time_s'] = metrics['sequence_time_ns'] / 1_000_000_000
metrics = metrics.drop('sequence_time_ns', axis=1)
metrics

In [ ]:
pathlib.Path('output_analysis').mkdir(parents=True, exist_ok=True)

# Diffuse GI passes, as defined in breda-global-illumination (incl. its clipmap cache).
DIFFUSE_GI_PASSES = [
    # ray generation + tracing
    'prepare-diffuse-rays',
    'trace-diffuse-rays',
    'trace-diffuse-nee-rays',
    'trace-spatial-validation-rays-inline',
    'resolve-diffuse-rays',
    # ReSTIR
    'diffuse-restir-temporal',
    'diffuse-restir-spatial',
    'diffuse-restir-patch-reservoir',
    'diffuse-restir-resolve',
    # denoising
    'diffuse-temporal',
    # clipmap radiance cache
    'update-clipmap',
    'update-lifetimes',
    'trace-cache-diffuse-rays-low',
    'build-spherical-harmonics',
]

# Skip the first seconds of the benchmark (startup noise / warmup)
START_TIME_S = 5

# Transparent background with light text/grid, for embedding on a dark page
TEXT_COLOR = 'white'

# Headroom above the stack so the legend does not touch it
HEADROOM_FACTOR = 1.35

# Fixed color per pass for every plot, assigned by the adreno file's
# biggest-to-smallest stack order, so the same pass has the same color everywhere.
COLOR_REFERENCE_FILE = files[-1]
reference_data = full_dataset.loc[COLOR_REFERENCE_FILE]
reference_order = (
    reference_data[[c for c in DIFFUSE_GI_PASSES if c in reference_data.columns]]
    .infer_objects(copy=False)
    .interpolate(method='linear')
    .fillna(0)
    .mean()
    .sort_values(ascending=False)
    .index
)
PASS_COLORS = {label: plt.get_cmap('tab20')(i) for i, label in enumerate(reference_order)}

# Every plot starts at the same time: the first file's (xclipse) first frame
# plus the warmup skip, so the x axes line up between plots
x_start = full_dataset.loc[files[0], 'sequence_time_ns'].min() / 1_000_000_000 + START_TIME_S

# Prepare the stack data for every file first, so all plots can share one scale.
prepared = {}
for file_name in files:
    file_data = full_dataset.loc[file_name]

    # x axis: benchmark timeline in seconds
    x = file_data['sequence_time_ns'] / 1_000_000_000

    # Only the diffuse GI passes, interpolated and gap-filled,
    # then smoothed with a centered 9-frame mean filter to reduce frame-to-frame noise
    metric_cols = [c for c in DIFFUSE_GI_PASSES if c in file_data.columns]
    stack_data = (
        file_data[metric_cols]
        .infer_objects(copy=False)
        .interpolate(method='linear')
        .fillna(0)
        .rolling(window=9, center=True, min_periods=1)
        .mean()
    )

    # Drop the frames before the shared start time
    mask = x >= x_start
    prepared[file_name] = (x[mask], stack_data[mask])

# Same axis limits for every plot: shared start/end on x, and on y the
# tallest stack across all files plus headroom
x_end = max(x.max() for x, _ in prepared.values())
y_limit = max(stack.sum(axis=1).max() for _, stack in prepared.values()) * HEADROOM_FACTOR - 3

# Plot the diffuse GI passes stacked on top of each other, one stackplot per input
# file, so we can see how the total diffuse GI time is composed over the benchmark.
for file_name, (x, stack_data) in prepared.items():
    # Biggest contributors at the bottom, each pass in its fixed color
    labels = stack_data.mean().sort_values(ascending=False).index.tolist()
    plot_data = stack_data[labels]
    colors = [PASS_COLORS[label] for label in labels]

    fig, ax = plt.subplots(figsize=(16, 9))
    fig.patch.set_alpha(0)
    ax.set_facecolor('none')
    ax.stackplot(x, plot_data.T.values, labels=labels, colors=colors)
    ax.set_xlabel('benchmark timeline in seconds', fontsize=18, color=TEXT_COLOR)
    ax.set_ylabel('shader execution time in ms', fontsize=18, color=TEXT_COLOR)
    ax.tick_params(labelsize=16, colors=TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_color(TEXT_COLOR)
    ax.set_xlim(x_start, x_end)
    ax.set_ylim(0, y_limit)
    ax.grid(True, color=TEXT_COLOR, alpha=0.2)

    # Legend inside the graph, reversed so its order matches the
    # top-to-bottom order of the stack
    handles, leg_labels = ax.get_legend_handles_labels()
    ax.legend(
        handles[::-1], leg_labels[::-1], loc='upper left', ncol=2, fontsize=14,
        facecolor='none', edgecolor=TEXT_COLOR, labelcolor=TEXT_COLOR,
    )

    fig.tight_layout()
    fig.savefig(f'output_analysis/{file_name}_diffuse_gi_stackplot.png', bbox_inches='tight', transparent=True)


In [ ]:
# Mean time per diffuse GI pass across the entire timeline, one column per input file
mean_table = pd.DataFrame({
    file_name: full_dataset.loc[file_name][
        [c for c in DIFFUSE_GI_PASSES if c in full_dataset.loc[file_name].columns]
    ]
    .infer_objects(copy=False)
    .interpolate(method='linear')
    .fillna(0)
    .mean()
    for file_name in files
})

mean_table = mean_table.sort_values(files[0], ascending=False)
mean_table.loc['total'] = mean_table.sum()
mean_table.to_csv('output_analysis/diffuse_gi_mean_table.csv')
mean_table.round(3)
